In [0]:
%run /Workspace/etl_spotify/Notebooks/01_bronze_extract_spotify.py

In [0]:
##########################################
           # READING ALL ITEMS #
##########################################


import json
from pyspark.sql.functions import count,when, col, size, lit, explode_outer, desc, count, sum, avg, round


path_all_items = "/Volumes/spotify_pipeline/spotify/raw/all_items.json"
path_artist_genre = "/Volumes/spotify_pipeline/spotify/raw/genre_by_artists.json"

# all_items
with open(path_all_items, "w") as f:
    json.dump(all_items, f)
df_all_items = spark.read.option("multiline", "true").json(path_all_items)

# path_artist_genre
with open(path_artist_genre, "w") as f:
    json.dump(genre_by_artists, f)
df_genre_by_artists = spark.read.option("multiline", "true").json(path_artist_genre)



In [0]:
##########################################
           # Data Preparation #
##########################################

# Tracks
df_tracks = df_all_items.selectExpr(
    "track.id as track_id",
    "track.name as track_name",
    "added_at as date_added"
)
df_tracks.createOrReplaceTempView("tmp_tracks")

# Track meta
df_track_meta = df_all_items.selectExpr(
    "track.id as track_id",
    "track.disc_number",
    "track.duration_ms",
    "track.explicit",
    "track.is_playable",
    "track.popularity",
    "track.preview_url",
    "track.track_number",
    "track.type",
    "track.uri"
)
df_track_meta.createOrReplaceTempView("tmp_track_meta")

# track market
df_track_markets = df_all_items.selectExpr(
    "track.id as track_id",
    "explode(track.available_markets) as market"
)
df_track_markets.createOrReplaceTempView("tmp_track_markets")

# external ids
df_external_ids = df_all_items.selectExpr(
    "track.id as track_id",
    "track.external_ids.isrc as isrc"
)
df_external_ids.createOrReplaceTempView("tmp_external_ids")

# Albums
df_albums = df_all_items.selectExpr(
    "track.id as track_id",
    "track.album.id as album_id",
    "track.album.name as album_name",
    "track.album.release_date as album_release_date",
    "get(filter(track.album.images, x -> x.width = 640), 0).url as album_art"
)
df_albums.createOrReplaceTempView("tmp_albums")

# album market
df_album_markets = df_all_items.selectExpr(
    "track.album.id as album_id",
    "explode(track.album.available_markets) as market"
)
df_album_markets.createOrReplaceTempView("tmp_album_markets")


# artists
df_artists = df_all_items.selectExpr(
    "track.id as track_id",
    "explode(track.artists) as artist"
).selectExpr(
    "track_id",
    "artist.id as artist_id",
    "artist.name as artist_name",
    "artist.href as artist_href",
    "artist.external_urls.spotify as artist_spotify_url"
)
df_artists.createOrReplaceTempView("tmp_artists")


# artists to genre
df_genartists = df_genre_by_artists.select("artist_id", explode_outer("genres").alias("genres"))
df_genartists.createOrReplaceTempView("tmp_genartists")

# User master data with track features
df_sp_master_data_with_track_features = spark.read.option("header", True).csv("/Volumes/spotify_pipeline/spotify/raw/sp_master_data_with_track_features.csv")
df_sp_master_data_with_track_features.createOrReplaceTempView("tmp_sp_master")

# Super track features
df_super_tracks_features = spark.read.option("header", True).csv("/Volumes/spotify_pipeline/spotify/raw/tracks_features.csv")
df_super_tracks_features.createOrReplaceTempView("tmp_tracks_features")



In [0]:
df_super_tracks_features

In [0]:
%sql

/* *****************************************************
           SQL Practical Questions/Answers MEDIUM


-- List all tracks that have more than one artist. 
select track_id,  count(artist_id) from tmp_df_artists group by track_id  having count
 > 1 order by count(artist_id) desc;

-- Find the most popular track in each album.
with cte as (
select t.track_id, t.track_name, a.album_name, m.popularity,
row_number() over (partition by a.album_id order by m.popularity desc) as popular_song_per_album
from tmp_tracks t
join tmp_track_meta m on m.track_id = t.track_id
join tmp_albums a on a.track_id = t.track_id
order by a.album_name, m.popularity desc, t.track_id
) 
select track_name, album_name from cte where popular_song_per_album = 1;

-- Calculate the average duration of tracks per album in minutes.
select a.album_id, a.album_name, round(avg(m.duration_ms)/1000/60, 2) as total_duration
from tmp_albums a
join tmp_track_meta m on m.track_id = a.track_id
group by  a.album_id, a.album_name
order by avg(m.duration_ms) desc;

-- List all albums that are available in the 'US' market.
select a.album_name, a.album_id, m.market
from tmp_albums a
join tmp_track_markets m on m.track_id = a.track_id
where market like 'US'
order by a.album_name;

-- Show all tracks added after January 1, 2025.
select * from tmp_tracks where date_added > '2025-01-01';

-- Identify the top 5 artists with the most tracks in the dataset.
select artist_id, artist_name, count(track_id) as totaloccurence
from tmp_artists
group by artist_id, artist_name
order by totaloccurence desc
limit 5;

-- Find all tracks that are marked as explicit.
select track_id from tmp_track_meta
where explicit = true;

--List tracks that are playable but have popularity less than 20.
select t.track_id, t.track_name 
from tmp_track_meta m
join tmp_tracks t on t.track_id = m.track_id
where is_playable = true and popularity < 20;

--Identify albums that contain more than 10 tracks.
select a.album_name, a.album_id, count(a.track_id)
from tmp_albums a
group by a.album_id, a.album_name
having count(a.track_id) > 10;

-- Find tracks that are available in both 'US' and 'GB' markets.
select t.track_id from (
select m.track_id
from tmp_track_markets m
where market in ('US')) as t
join tmp_track_markets m2 on m2.track_id = t.track_id
where m2.market = 'GB';
****************************************************** */


In [0]:
%sql

/* *****************************************************
           SQL Practical Questions/Answers

HARD:
--Find all tracks where the same artist appears on multiple albums and list the track IDs, artist name, and album names.
select b.track_id,a.artist_name, b.album_name, b.album_id
from (
select distinct t.artist_name, t.artist_id from (
select t.track_id, ar.artist_name, ar.artist_id, row_number() over (partition by artist_name, album_id order by album_id) as rw,
t.album_name, t.album_id
from tmp_albums t
join tmp_artists ar on ar.track_id = t.track_id
where t.track_id is not null) as t
where rw > 1
) as t1
join tmp_artists a on a.artist_id = t1.artist_id
join tmp_albums b on b.track_id = a.track_id
where album_name is not null
order by artist_name
--
select 
ar.artist_name , count(distinct t.album_id) as album_count
from tmp_albums t
join tmp_artists ar on ar.track_id = t.track_id
group by ar.artist_name 
--

--Find albums where the most popular track is also the longest track.
select * from (
select a.album_id, 
a.album_name, 
tm.track_id, 
tm.duration_ms, 
row_number() over (partition by album_name order by tm.duration_ms desc) as longest,
row_number() over (partition by album_name order by tm.popularity desc) as mostpopular
from tmp_track_meta tm 
join tmp_albums a on a.track_id = tm.track_id) as t
--where album_name like '%Toxicity%'
where longest = mostpopular

--Identify artists who have tracks in multiple genres and show the number of unique genres per artist.
select t.artist_id, 
t.artist_name,
genres 
from (
select 
a.artist_id,
a.artist_name, 
count(distinct ga.genres) as genre_ct 
from tmp_genartists ga
join tmp_artists a on a.artist_id = ga.artist_id
where 1=1
and genres is not null
--and a.artist_name like '%Beyoncé%'

group by a.artist_name, a.artist_id) as t
join tmp_genartists gaa on gaa.artist_id = t.artist_id
where genre_ct > 1
order by t.artist_name

select album_name, sum(duration_ms)
from tmp_track_meta t1 
join tmp_albums b on b.track_id = t1.track_id
where album_name like '%Endless%'
group by album_name 

****************************************************** */
select album_name, sum(duration_ms)
from tmp_track_meta t1 
join tmp_albums b on b.track_id = t1.track_id
where album_name like '%Endless%'
group by album_name 



In [0]:
'''
PySpark Questions (Beginner Level)

1) Display schema of df_tracks and show the first 5 rows.
df_tracks.printSchema()
df_tracks.show(5)

2) Select only the track_id and track_name columns from df_tracks.
df_tracks.select("track_id", "track_name").show(5)

3) Filter tracks added after "2025-01-01" from df_tracks.
display(df_tracks.filter(col("date_added") > "2025-01-01"))

4) Find all explicit tracks from df_track_meta.
display(df_track_meta.filter(col("explicit") == "True"))

5) Count how many tracks are marked as explicit (explicit == True).
display(df_track_meta.filter(col("explicit") == "True").count())

6) Find unique markets available in df_track_markets.
display(df_track_markets.select("market").distinct())

7) Join df_tracks and df_track_meta on track_id and show track_name, popularity, and duration_ms.
display(df_tracks.join(df_track_meta, "track_id", "inner").select("track_id", "track_name", "popularity", "duration_ms").orderBy(desc("track_name")))

8) Group by album_id in df_albums and count how many tracks belong to each album.
display(df_tracks.join(df_albums, "track_id", "inner").groupBy("album_id", "album_name").agg(count("*").alias("trackct")).orderBy(desc("trackct")))

9) Find the top 5 tracks by popularity using orderBy(desc("popularity")).
display(df_track_meta.join(df_tracks, "track_id", "inner").select("track_id", "track_name", "popularity").orderBy(desc("popularity")).limit(5))

10) Songs that are popular (89+)
display(df_track_meta.join(df_tracks, "track_id", "inner").select("track_id", "track_name", "popularity").filter(col("popularity") > 85).orderBy(desc("popularity")))

11) Add a new column to df_track_meta that converts duration_ms into minutes (rounded to 2 decimals).
(Hint: divide by 1000 * 60)
display(
    df_track_meta.withColumn("duration_mins", round(col("duration_ms") / 1000 / 60, 2))
)

12) Show the average popularity of tracks grouped by each album.
display(df_albums.join(df_track_meta, "track_id", "inner").groupBy("album_id", "album_name").agg(avg(col("popularity")).alias("ct"))
.orderBy(desc("ct")))

13) Find the total duration (in minutes) of all tracks per album.
display(
    df_albums
        .join(df_track_meta, "track_id", "inner")
        .groupBy("album_id", "album_name")
        .agg(sum("duration_ms").alias("ct"))
       .filter(col("album_name").like("%Endless%"))
)

14) Identify the top 10 artists with the highest number of explicit songs.
display(
    df_artists.join(df_track_meta, "track_id", "inner")
    .filter(col("explicit") == True)
    .groupBy("artist_id", "artist_name")
    .agg(count("track_id").alias("total"))
    .orderBy(desc("total"))
    .limit(10)
)

15) Display a histogram of track popularity to visualize the popularity distribution.
NO

16) List the top 10 longest tracks in minutes.
display(
    df_tracks.join(df_track_meta, "track_id", "inner")
    .select("track_id", "track_name", round(col("duration_ms")/1000/60, 2).alias("duration_mins"))
    .orderBy(desc("duration_mins"))
    .limit(10)
)

17) Find the number of tracks available in each market.
display(
    df_tracks.join(df_track_markets, "track_id", "inner")
    #.filter(col("market").like("%IN%"))
    .groupBy("market")
    .agg(count("track_id").alias("number_of_tracks"))
    .orderBy(desc("number_of_tracks"))
)

18) For every album, calculate the ratio of explicit to total tracks.

display(
    df_albums.join(df_track_meta, "track_id", "inner")
    .groupBy("album_name")
    .agg(
        count("track_id").alias("total_tracks"),
        count(when(col("explicit") == True, 1)).alias("explicit_tracks")
    )
    .withColumn(
        "explicit_ratio", 
        round(col("explicit_tracks") / col("total_tracks") * 100, 2)
    )
    .orderBy(desc("explicit_ratio"))

19) Average track duration per artist
display(
    df_artists.join(df_track_meta, "track_id", "inner")
    .groupBy("artist_name")
    .agg(
        round(
        sum("duration_ms")/(1000)/(60), 2).alias("total_duration"))
    .orderBy(desc("artist_name"))
    )


20) Top 5 albums with the highest average popularity

display(
    df_albums.join(df_track_meta, "track_id", "inner")
    .groupBy("album_name")
    .agg(
        round(
        avg("popularity"), 2).alias("avg_pop"))
    .orderBy(desc("avg_pop"))
    .limit(5)
    )

21) Count of tracks per genre
display(
    df_genartists.join(df_artists, "artist_id", "inner")
    .groupBy("genres")
    .agg(
        round(
        count("track_id"), 2).alias("tracks_per_genre"))
    .orderBy(desc("tracks_per_genre"))
   # .limit(5)
    )

22) Artists with more than 50% of tracks explicit
display(
    df_artists.join(df_track_meta, "track_id", "inner")
    .groupBy("artist_name")
    .agg(
        count("track_id").alias("total_tracks"),
        count(when(col("explicit") == True, 1)).alias("explicit_tracks")
    )
    .withColumn(
        "explicit_ratio", 
        round(col("explicit_tracks") / col("total_tracks") * 100, 2)
    )
    .filter(col("explicit_ratio") > 50)
    .orderBy(desc("explicit_ratio")))

23) Tracks longer than 10 minutes

display(
    df_tracks
        .join(df_track_meta, "track_id", "inner")
        .withColumn("duration_minutes", round(col("duration_ms") / 1000 / 60, 2))
        .filter(col("duration_minutes") > 10)
        .orderBy(desc("track_name"))
)

24) Top 5 markets with most explicit tracks
display(
    df_track_markets.join(df_track_meta, "track_id", "inner")
    .groupBy("market")
    .agg(
        count(when(col("explicit") == True, 1)).alias("explicit_tracks")
    )
    .orderBy(desc("explicit_tracks"), "market")
    .limit(5))

25) Popular vs non-popular ratio per artist

display(
    df_artists.join(df_track_meta, "track_id", "inner")
    .groupBy("artist_id")
    .agg(
        count("track_id").alias("total"),
        count(when(col("popularity") > 80, 1)).alias("popular_tracks"),
        count(when(col("popularity") <= 80, 1)).alias("unpopular_tracks")
    )
    .withColumn(
        "Popularity_ratio", 
        round(
            when(
                    col("unpopular_tracks") != 0, col("popular_tracks")/ col("unpopular_tracks")).otherwise(None), 2     )) 
    .orderBy(desc("artist_id"))
  )
  
last) For df_tracks, find out how many null (missing) values exist in each column.



'''



In [0]:


'''
spark = (
    SparkSession
        .builder
        .appName("Practice")
        .getOrCreate()
)

display(spark
        .sql("""
             
        with cte as (select 
            genres,
            DATE_FORMAT(date_added, 'yyyy-MM') as yearmonth,           
            round(avg(popularity), 2) as avg_pop,
           count(t.track_id) as total_tracks,
           round(sum(m.duration_ms/1000/60), 2) as tot_duration_mins
        from tmp_tracks t
        join tmp_artists a on a.track_id = t.track_id
        join tmp_genartists ag on ag.artist_id = a.artist_id
        join tmp_track_meta m on m.track_id = t.track_id
        where  genres is not null
        group by genres,yearmonth) 
        select 
            genres,
            yearmonth,
            avg_pop,
            total_tracks,
            tot_duration_mins,
            sum(total_tracks) over (partition by genres order by yearmonth) as cumulative_tracks,
            dense_rank() over (partition by  yearmonth order by avg_pop desc) as ran
        from cte order by yearmonth 
                  """))






'''

df_yearmonth = (
    df_tracks
    .withColumn("dateformat", date_format(col("date_added"), "yyyy-MM"))
    .groupBy("dateformat")
    .agg(count(col("track_id")).alias("total"))
    .orderBy("dateformat")
)


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8468506243528577>, line 43
      1 '''
      2 spark = (
      3     SparkSession
   (...)
     39 
     40 '''
     42 df_yearmonth = (
---> 43     df_tracks
     44     .withColumn("dateformat", date_format(col("date_added"), "yyyy-MM"))
     45     .groupBy("dateformat")
     46     .agg(count(col("track_id")).alias("total"))
     47     .orderBy("dateformat")
     48 )

NameError: name 'df_tracks' is not defined

In [0]:
%sql

SHOW CATALOGS;
SHOW SCHEMAS;

databaseName
default
information_schema
yt2sp
